# Aquí veremos distintas formas de comparar mapas, ya sea que provengan de un mismo método de sampleo, o bien, entre mecanismos.

In [1]:
import os
import pandas as pd


ruta_resultados = r"C:\Tesis\Códigos\resultados"


def df_a_labels(df):
    df = df[df["x"] == 1].copy()

    asignacion = dict(zip(df["i"], df["j"]))

    centros = sorted(set(asignacion.values()))
    mapa_centros = {c: idx for idx, c in enumerate(centros)}

    comunas = sorted(asignacion.keys())
    labels = [mapa_centros[asignacion[i]] for i in comunas]

    return labels, comunas


def cargar_labels_desde_resultados(ruta_resultados):
    mapas = {}
    comunas_base = None

    for carpeta in sorted(os.listdir(ruta_resultados)):
        path_carpeta = os.path.join(ruta_resultados, carpeta)

        if not os.path.isdir(path_carpeta):
            continue

        archivo = f"modelo_{carpeta}_asignaciones.csv"
        path_archivo = os.path.join(path_carpeta, archivo)

        if not os.path.exists(path_archivo):
            continue

        df = pd.read_csv(path_archivo)
        labels, comunas = df_a_labels(df)

        if comunas_base is None:
            comunas_base = comunas
        elif comunas != comunas_base:
            raise ValueError(f"Las comunas de {carpeta} no coinciden con las del resto")

        mapas[carpeta] = labels

    return mapas, comunas_base

In [2]:
from funciones_metricas import matriz_metricas, comunas_inestables, estabilidad_comuna, comunas_inestables_nombres

In [3]:
mapas, comunas = cargar_labels_desde_resultados(ruta_resultados)

In [4]:
idx = comunas.index("arica")
mapas["t_001"][idx]

3

In [5]:
matriz_ari = matriz_metricas(mapas, metrica="ari")
matriz_nmi = matriz_metricas(mapas, metrica="nmi")
matriz_vi  = matriz_metricas(mapas, metrica="vi")

In [6]:
matriz_ari

,t_001,t_002,t_003,t_004,t_005,t_006,t_007,t_008,t_009,t_010,...,t_091,t_092,t_093,t_094,t_095,t_096,t_097,t_098,t_099,t_100
t_001,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000
t_002,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000
t_003,0.952613,0.952613,1.000000,0.951446,0.952613,0.952613,0.952613,0.955742,0.952613,0.952613,...,0.951446,0.937738,0.952613,0.952613,0.951446,0.952613,0.951446,0.952613,0.952613,0.952613
t_004,0.995173,0.995173,0.951446,1.000000,0.995173,0.995173,0.995173,0.986678,0.995173,0.995173,...,1.000000,0.974273,0.995173,0.995173,1.000000,0.995173,1.000000,0.995173,0.995173,0.995173
t_005,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
t_096,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000
t_097,0.995173,0.995173,0.951446,1.000000,0.995173,0.995173,0.995173,0.986678,0.995173,0.995173,...,1.000000,0.974273,0.995173,0.995173,1.000000,0.995173,1.000000,0.995173,0.995173,0.995173
t_098,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000
t_099,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000


In [7]:
import numpy as np

In [8]:
def resumen_matriz(M):
    valores = M.values
    valores = valores[~np.eye(valores.shape[0], dtype=bool)]
    return {
        "mean": valores.mean(),
        "std": valores.std(),
        "min": valores.min(),
        "max": valores.max()
    }

resumen_ari = resumen_matriz(matriz_ari)
resumen_nmi = resumen_matriz(matriz_nmi)
resumen_vi  = resumen_matriz(matriz_vi)

In [9]:
resumen_ari

{'mean': np.float64(0.9838675752148585),
 'std': np.float64(0.018408153260988207),
 'min': np.float64(0.9377378752551687),
 'max': np.float64(1.0)}

In [10]:
resumen_nmi

{'mean': np.float64(0.9805550315251441),
 'std': np.float64(0.020290122030424188),
 'min': np.float64(0.941093068221218),
 'max': np.float64(1.0)}

In [11]:
resumen_vi

{'mean': np.float64(0.11951462797250685),
 'std': np.float64(0.12472648059717184),
 'min': np.float64(0.0),
 'max': np.float64(0.36231215404058315)}

In [12]:
inestables = comunas_inestables(mapas)
len(inestables)

207

In [13]:
nombres_inesatables = comunas_inestables_nombres(mapas, comunas)

In [17]:
len(nombres_inesatables)

207

In [14]:
nombres_inesatables

{'alhue': {17: 50, 13: 6, 16: 32, 14: 7, 11: 5},
 'ancud': {15: 62, 14: 38},
 'andacollo': {16: 62, 15: 38},
 'antartica': {22: 73, 21: 27},
 'arauco': {6: 99, 13: 1},
 'aysen': {20: 73, 19: 27},
 'buin': {17: 57, 13: 4, 16: 34, 11: 5},
 'bulnes': {5: 98, 4: 2},
 'cabo de hornos': {22: 73, 21: 27},
 'calbuco': {15: 62, 14: 38},
 'caldera': {7: 96, 23: 4},
 'calera de tango': {4: 98, 16: 2},
 'camina': {10: 96, 9: 4},
 'canela': {8: 96, 7: 4},
 'castro': {15: 62, 14: 38},
 'cerrillos': {13: 81, 11: 6, 12: 13},
 'cerro navia': {13: 78, 4: 2, 21: 11, 14: 7, 5: 2},
 'chaiten': {15: 62, 14: 38},
 'chanaral': {7: 96, 23: 4},
 'chepica': {19: 62, 18: 38},
 'chiguayante': {6: 99, 13: 1},
 'chile chico': {20: 73, 19: 27},
 'chillan': {5: 98, 4: 2},
 'chillan viejo': {5: 98, 4: 2},
 'chimbarongo': {19: 62, 18: 38},
 'chonchi': {15: 62, 14: 38},
 'cisnes': {20: 73, 19: 27},
 'cobquecura': {5: 98, 4: 2},
 'cochamo': {15: 62, 14: 38},
 'cochrane': {20: 73, 19: 27},
 'codegua': {19: 62, 18: 38},
 'c

In [18]:
mapas_estabilidad = estabilidad_comuna(mapas)

In [20]:
len(mapas_estabilidad)

346

In [21]:
mapas_estabilidad

[1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 0.62,
 0.62,
 1.0,
 0.73,
 1.0,
 1.0,
 0.99,
 1.0,
 0.73,
 0.57,
 0.98,
 1.0,
 0.73,
 1.0,
 1.0,
 0.62,
 0.96,
 1.0,
 0.98,
 1.0,
 1.0,
 0.96,
 0.96,
 1.0,
 1.0,
 1.0,
 1.0,
 0.62,
 1.0,
 1.0,
 0.81,
 0.78,
 0.62,
 0.96,
 1.0,
 0.62,
 0.99,
 0.73,
 0.98,
 0.98,
 0.62,
 1.0,
 0.62,
 0.73,
 0.98,
 0.62,
 0.73,
 0.62,
 0.98,
 0.98,
 0.62,
 1.0,
 0.96,
 0.78,
 1.0,
 0.62,
 0.62,
 0.99,
 0.68,
 1.0,
 1.0,
 1.0,
 0.96,
 0.96,
 0.99,
 0.49,
 0.73,
 1.0,
 1.0,
 0.55,
 0.62,
 1.0,
 1.0,
 1.0,
 1.0,
 0.62,
 0.96,
 0.62,
 0.96,
 0.98,
 0.57,
 1.0,
 1.0,
 1.0,
 1.0,
 0.81,
 1.0,
 1.0,
 1.0,
 0.62,
 0.62,
 0.62,
 0.96,
 1.0,
 1.0,
 1.0,
 0.62,
 0.73,
 1.0,
 0.62,
 1.0,
 0.99,
 0.99,
 0.96,
 0.96,
 0.62,
 0.62,
 0.73,
 0.96,
 0.57,
 1.0,
 1.0,
 0.82,
 1.0,
 0.62,
 0.91,
 0.49,
 0.62,
 1.0,
 0.91,
 0.82,
 0.96,
 0.76,
 0.96,
 0.73,
 0.73,
 1.0,
 0.8,
 0.55,
 0.62,
 0.62,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.62,
 0.62,
 1.0,
 0.62,
 0.81,
 0.49,
 0.62,
 1.0,
 1.0,
 1.0,
 1.0